# Informe Trabajo Práctico Proyesat - Programación 1
Este notebook presenta el análisis integral del catálogo de satélites utilizando una arquitectura modular del proyecto: 'catalogo.py', 'satelite.py' y 'analisis.py', combinados con las funciones de **Skyfield**

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import catalogo
from satelite import crear_satelites
import analisis

## 1. Carga y filtrado del catálogo
Cargamos los datos desde el archivo JSON utilizando las funciones de lectura y generamos los objetos 'Satelite', descartando aquellos con una altitud menor a 100km

In [2]:
# Ruta al archivo JSON de datos (ajustar si es necesario)
ruta_json = Path("datos/satelites.json")

# 1. Leer registros crudos del catálogo
registros = catalogo.leer_satelites(ruta_json)
print(f"Total de registros leídos: {len(registros)}")

# 2. Definir el momento actual en UTC para el cálculo de posiciones
momento_actual = datetime.now(timezone.utc)

# 3. Crear los objetos Satelite válidos (altitud >= 100 km)
satelites = crear_satelites(registros, momento_actual)
print(f"Satélites válidos analizados: {len(satelites)}")

Total de registros leídos: 16822
Satélites válidos analizados: 16822


## 2. Análisis Estadístico por gtupos y tipos de órbita
Estudiamos cómo se distribuyen los satélites y los clasificamos por el tipo de órbita (LEO, MEO, GEO, HEO)

In [8]:
conteo_grupos = catalogo.contar_por_grupo(registros)
print("Conteo de registros por grupo en el catálogo:")
for grupo_nombre, cantidad in conteo_grupos.items():
    print(f"  - {grupo_nombre}: {cantidad}")

print("\n--- Resumen Estadístico de Altitudes por Grupo ---")
resumen_grupos = analisis.resumen_por_grupo(satelites)
for grupo_nombre, stats in resumen_grupos.items():
    print(f"Grupo: {grupo_nombre}")
    print(f"  Cantidad activa: {stats['cantidad']}")
    print(f"  Altitud Mínima: {stats['minima']:.2f} km")
    print(f"  Altitud Media: {stats['media']:.2f} km")
    print(f"  Altitud Máxima: {stats['maxima']:.2f} km\n")

conteo_tipos = analisis.contar_por_tipo(satelites)
print("Distribución por tipo de órbita:")
for tipo_orb, cant in conteo_tipos.items():
    print(f"  - {tipo_orb}: {cant}")

Conteo de registros por grupo en el catálogo:
  - Otros: 4152
  - GPS: 40
  - Globalstar: 36
  - BeiDou: 55
  - Iridium: 80
  - OneWeb: 651
  - Starlink: 11160
  - Qianfan: 257
  - Kuiper: 391

--- Resumen Estadístico de Altitudes por Grupo ---
Grupo: Otros
  Cantidad activa: 4152
  Altitud Mínima: 105.32 km
  Altitud Media: 6173.03 km
  Altitud Máxima: 136076.75 km

Grupo: GPS
  Cantidad activa: 40
  Altitud Mínima: 19747.48 km
  Altitud Media: 20257.11 km
  Altitud Máxima: 21018.91 km

Grupo: Globalstar
  Cantidad activa: 36
  Altitud Mínima: 677.71 km
  Altitud Media: 1297.22 km
  Altitud Máxima: 1831.29 km

Grupo: BeiDou
  Cantidad activa: 55
  Altitud Mínima: 21510.68 km
  Altitud Media: 27485.11 km
  Altitud Máxima: 35998.96 km

Grupo: Iridium
  Cantidad activa: 80
  Altitud Mínima: 638.98 km
  Altitud Media: 776.71 km
  Altitud Máxima: 807.95 km

Grupo: OneWeb
  Cantidad activa: 651
  Altitud Mínima: 468.45 km
  Altitud Media: 1209.29 km
  Altitud Máxima: 1252.34 km

Grupo: Star

## 3. Identificación de órbitas extremas

Identificamos cuáles son los satélites que se encuentran en las posiciones más bajas y más altas del conjunto analizado

In [10]:
if satelites:
    s_bajo = analisis.mas_bajo(satelites)
    s_alto = analisis.mas_alto(satelites)
    
    print(f"Satélite más bajo: {s_bajo}")
    print(f"Satélite más alto: {s_alto}")

    print("\nTop 5 satélites más bajos:")
    top_bajos = analisis.top_n(satelites, 5, mas_altos=False)
    for s in top_bajos:
        print(f"  * {s}")
    
    print("\nTop 5 satélites más altos:")
    top_altos = analisis.top_n(satelites, 5, mas_altos=True)
    for s in top_altos:
        print(f"  * {s}")

Satélite más bajo: RASSVET-3 23 | LEO | 105.322 km
Satélite más alto: CXO | HEO | 136076.747 km

Top 5 satélites más bajos:
  * RASSVET-3 23 | LEO | 105.322 km
  * STARLINK-1036 | LEO | 117.818 km
  * IPERDRONE.0 | LEO | 126.216 km
  * STARLINK-1309 | LEO | 188.069 km
  * RASSVET-3 17 | LEO | 195.683 km

Top 5 satélites más altos:
  * CXO | HEO | 136076.747 km
  * SMILE | HEO | 109215.652 km
  * THEMIS A | HEO | 73035.384 km
  * THEMIS E | HEO | 69075.312 km
  * ASTRANIS UTILITYSAT | HEO | 62571.145 km


## 4. Análisis geoespacial (filtrado por zona)

Evaluamos qué satélites se encuentran sobre una región geográfica delimitada por coordenadas de longitud y latitud `(lon_min, lat_min, lon_max, lat_max)`

In [9]:
zona_estudio = (-80.0, -60.0, -30.0, 15.0)

satelites_en_zona = analisis.en_zona(satelites, zona_estudio)
print(f"Satélites encontrados en la zona: {len(satelites_en_zona)}")
for s in satelites_en_zona[:5]:  
    print(f"  - {s} | Lat: {s.latitud:.2f}°, Lon: {s.longitud:.2f}°")

Satélites encontrados en la zona: 1378
  - OSCAR 7 (AO-7) | LEO | 1463.961 km | Lat: -42.49°, Lon: -61.53°
  - PHASE 3B (AO-10) | MEO | 4167.854 km | Lat: 14.23°, Lon: -43.85°
  - TDRS 3 | GEO | 35616.816 km | Lat: 0.33°, Lon: -48.77°
  - TDRS 6 | GEO | 35813.513 km | Lat: -2.79°, Lon: -45.40°
  - SCD 1 | LEO | 763.864 km | Lat: -24.71°, Lon: -35.59°


## 5. Conclusiones

* La modularización del código facilitó la separación de responsabilidades: la lectura de archivos y clasificación en `catalogo.py`, el modelo físico y geométrico en `satelite.py`, y las métricas avanzadas en `analisis.py`
  
* Las constelaciones comerciales en órbita baja (LEO), como Starlink, dominan ampliamente la distribución numérica del catálogo actual

  
* El filtrado inicial de objetos menores a 100 km garantiza que el análisis orbital no se vea distorsionado por reentradas atmosféricas o artefactos numéricos